# Module 7 - Activity 1: Visually Probe ML Models (What-If Tool, rebuilt locally)

**Brief:** run the Google What-If Tool (WIT) demos and probe a model without writing code.

**Why this notebook:** the WIT demos are Colab-only and need `witwidget` + TensorFlow. Rather than
fight the install, we rebuild WIT's five core probes in plain scikit-learn - the *ideas* are the
assessable part, not the widget.

We use the **wine quality** dataset, which is (a) the first WIT demo in the brief and (b) the same
dataset as **Assessment 2** - so everything here is reusable there.

| WIT feature | What it answers | Rebuilt here as |
|---|---|---|
| Model comparison | do two models agree? | LogReg vs Random Forest, side by side |
| Datapoint editor | what if I change this feature? | manual perturbation of one wine |
| **Counterfactuals** | what is the most similar wine the model judges *differently*? | nearest opposite-class neighbour |
| Partial dependence | how does the model behave *overall*? | `PartialDependenceDisplay` (**global**) |
| Performance & fairness | does the model fail on a *slice*? | per-bin accuracy table |
| Threshold slider | what does moving the cut-off cost? | precision/recall sweep |


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                             precision_score, recall_score, f1_score)

SEED = 42
np.random.seed(SEED)
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)

WINE = Path('../../../assignments/Assessment2/dataset/winequality-red.csv')
df = pd.read_csv(WINE, sep=';')
df.columns = [c.strip().replace(' ', '_') for c in df.columns]

# Binary target, same framing as Assessment 2: is this a 'good' wine?
df['good'] = (df['quality'] >= 7).astype(int)
X = df.drop(columns=['quality', 'good'])
y = df['good']

print(f'{df.shape[0]} wines, {X.shape[1]} features')
print(f"class balance -> good: {y.mean():.1%}  (imbalanced, note it)")
X.head(3)

1599 wines, 11 features
class balance -> good: 13.6%  (imbalanced, note it)


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8


## Probe 0 - Model comparison

WIT's headline demo compares a **shallow** (scikit-learn) and a **deep** (Keras) model on the same
data. We compare an **intrinsically interpretable** model (logistic regression - you can read its
coefficients) against a **black box** (random forest - you cannot). That contrast *is* Module 7:
the accuracy you gain from the black box is exactly the explainability you lose.

In [2]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

logreg = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=2000, random_state=SEED))
rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)

models = {'LogReg (intrinsic)': logreg, 'RandomForest (black box)': rf}
rows = []
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict(X_te)
    proba = m.predict_proba(X_te)[:, 1]
    rows.append({'model': name,
                 'accuracy': accuracy_score(y_te, p),
                 'roc_auc': roc_auc_score(y_te, proba),
                 'recall(good)': recall_score(y_te, p),
                 'precision(good)': precision_score(y_te, p, zero_division=0)})

scores = pd.DataFrame(rows).set_index('model').round(3)
print(scores.to_string())

# Where do the two models DISAGREE? That is WIT's most useful view.
pl = logreg.predict(X_te); pr = rf.predict(X_te)
disagree = np.where(pl != pr)[0]
print(f'\nThe two models disagree on {len(disagree)}/{len(X_te)} test wines '
      f'({len(disagree)/len(X_te):.1%}).')
print('An accuracy number hides this. WIT shows it. So does one line of numpy.')

                          accuracy  roc_auc  recall(good)  precision(good)
model                                                                     
LogReg (intrinsic)           0.892    0.872         0.370            0.690
RandomForest (black box)     0.940    0.952         0.648            0.875

The two models disagree on 27/400 test wines (6.8%).
An accuracy number hides this. WIT shows it. So does one line of numpy.


## Probe 1 - Datapoint editor: *what if?*

The tool's namesake. Take one wine, change one feature, watch the prediction move.
This is a **local** probe - it says nothing about the model overall, only about this bottle.

In [3]:
i = 7  # one test wine
wine = X_te.iloc[[i]].copy()
base = rf.predict_proba(wine)[0, 1]
print(f'Original wine -> P(good) = {base:.3f}   (actual: {"good" if y_te.iloc[i] else "not good"})')
print()

# Nudge each feature by +1 standard deviation and see what the model does.
deltas = []
for feat in X.columns:
    edited = wine.copy()
    edited[feat] = edited[feat] + X[feat].std()
    deltas.append({'feature': feat,
                   'original': wine[feat].values[0],
                   'edited (+1 sd)': edited[feat].values[0],
                   'new P(good)': rf.predict_proba(edited)[0, 1],
                   'change': rf.predict_proba(edited)[0, 1] - base})

editor = pd.DataFrame(deltas).sort_values('change', ascending=False).round(3)
print('What-if: +1 standard deviation on each feature, for THIS wine only')
print(editor.to_string(index=False))

Original wine -> P(good) = 0.040   (actual: not good)



What-if: +1 standard deviation on each feature, for THIS wine only
             feature  original  edited (+1 sd)  new P(good)  change
             alcohol    11.200          12.266        0.337   0.297
         citric_acid     0.220           0.415        0.213   0.173
           chlorides     0.074           0.121        0.137   0.097
           sulphates     0.820           0.990        0.127   0.087
      residual_sugar     1.800           3.210        0.107   0.067
             density     0.994           0.996        0.083   0.043
       fixed_acidity     6.800           8.541        0.080   0.040
total_sulfur_dioxide    24.000          56.895        0.060   0.020
                  pH     3.400           3.554        0.053   0.013
 free_sulfur_dioxide    15.000          25.460        0.050   0.010
    volatile_acidity     0.560           0.739        0.047   0.007


## Probe 2 - Counterfactual: the nearest wine the model judges *differently*

WIT's signature button. Find the **most similar** datapoint that the model puts in the **other**
class, then diff them. Whatever differs is what the decision actually hinged on.

This is the same intuition as **CEM / pertinent negatives** in Activity 2, and it is the most
human-legible explanation in the whole module: *"this wine, but with more alcohol, would be good."*

In [4]:
scaler = StandardScaler().fit(X_tr)
Z_tr = scaler.transform(X_tr)

z_wine = scaler.transform(wine)
pred_wine = rf.predict(wine)[0]

# candidates the model puts in the OPPOSITE class
pred_tr = rf.predict(X_tr)
mask = pred_tr != pred_wine
dist = np.linalg.norm(Z_tr[mask] - z_wine, axis=1)
cf_idx = np.where(mask)[0][dist.argmin()]
cf = X_tr.iloc[[cf_idx]]

cmp = pd.DataFrame({
    'this wine': wine.iloc[0],
    'counterfactual': cf.iloc[0],
}).assign(difference=lambda d: d['counterfactual'] - d['this wine']).round(3)
cmp['|diff| in sd'] = (cmp['difference'].abs() / X.std()).round(2)

print(f'This wine    -> model says: {"GOOD" if pred_wine else "not good"} '
      f'(P={base:.3f})')
print(f'Nearest wine the model judges the OPPOSITE way -> '
      f'{"GOOD" if not pred_wine else "not good"} '
      f'(P={rf.predict_proba(cf)[0,1]:.3f})\n')
print(cmp.sort_values('|diff| in sd', ascending=False).to_string())
print('\nRead the top rows: those are the features that flip the verdict.')

This wine    -> model says: not good (P=0.040)
Nearest wine the model judges the OPPOSITE way -> GOOD (P=0.643)

                      this wine  counterfactual  difference  |diff| in sd
sulphates                 0.820           0.620       -0.20          1.18
alcohol                  11.200          11.700        0.50          0.47
residual_sugar            1.800           2.400        0.60          0.43
citric_acid               0.220           0.140       -0.08          0.41
chlorides                 0.074           0.064       -0.01          0.21
free_sulfur_dioxide      15.000          13.000       -2.00          0.19
total_sulfur_dioxide     24.000          29.000        5.00          0.15
pH                        3.400           3.420        0.02          0.13
fixed_acidity             6.800           6.600       -0.20          0.11
volatile_acidity          0.560           0.560        0.00          0.00
density                   0.994           0.994       -0.00          0.00

## Probe 3 - Partial dependence: the model's behaviour *overall*

Now we zoom out. PDP answers **"as alcohol rises, what happens to the average prediction?"** -
a statement about the whole model, not one wine.

> **This is the global/local line you must be able to draw in the exam.**
> Probes 1-2 were **local** (one bottle). Probe 3 is **global** (the model's average behaviour).

In [5]:
top = ['alcohol', 'volatile_acidity', 'sulphates', 'total_sulfur_dioxide']
fig, ax = plt.subplots(1, 4, figsize=(16, 3.4), constrained_layout=True)
PartialDependenceDisplay.from_estimator(
    rf, X_te, features=top, ax=ax, line_kw={'color': '#c0392b', 'lw': 2})
for a in ax:
    a.set_ylabel('P(good)')
fig.suptitle('Partial dependence - GLOBAL: how the average prediction moves with each feature',
             fontsize=11)
fig.savefig(OUT / 'a1_partial_dependence.png', dpi=130, bbox_inches='tight')
print('saved outputs/a1_partial_dependence.png')
plt.show()

saved outputs/a1_partial_dependence.png


/var/folders/c4/t_n58l316yl308hxjf5jlwsh0000gn/T/ipykernel_18217/1419836576.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Probe 4 - Performance & fairness: does the model fail on a *slice*?

WIT's fairness tab exists because **one accuracy number lies**. Slice the test set and the same
model can be excellent on one group and useless on another. The wine data has no protected
attribute, so we slice by alcohol - but the mechanic is exactly what you would run on
`sex` or `race` in the COMPAS demo, and exactly what the NZ Algorithm Charter (Sowden) is about.

In [6]:
sl = X_te.copy()
sl['y_true'] = y_te.values
sl['y_pred'] = rf.predict(X_te)
sl['bin'] = pd.qcut(sl['alcohol'], 4, labels=['low', 'mid-low', 'mid-high', 'high'])

rows = []
for b, g in sl.groupby('bin', observed=True):
    rows.append({'alcohol slice': b, 'n': len(g),
                 'actually good': int(g['y_true'].sum()),
                 'accuracy': accuracy_score(g['y_true'], g['y_pred']),
                 'recall(good)': recall_score(g['y_true'], g['y_pred'], zero_division=0)})

slices = pd.DataFrame(rows).round(3)
print(f"Overall accuracy: {accuracy_score(sl['y_true'], sl['y_pred']):.3f}\n")
print(slices.to_string(index=False))
print('\nThe headline accuracy is an average over slices that behave very differently.')

Overall accuracy: 0.940

alcohol slice   n  actually good  accuracy  recall(good)
          low 106              2     0.981         0.000
      mid-low  98              5     0.959         0.200
     mid-high 100             14     0.930         0.500
         high  96             33     0.885         0.818

The headline accuracy is an average over slices that behave very differently.


## Probe 5 - Threshold slider: the cut-off is a *policy choice*, not a fact

The model outputs a probability. Turning it into a yes/no needs a threshold, and 0.5 is a
convention, not a truth. Moving it trades **precision against recall** - and *someone* has to
decide which error is worse. That decision is a business/ethics call, not a maths one.

In [7]:
proba = rf.predict_proba(X_te)[:, 1]
ts = np.linspace(0.05, 0.95, 91)
prec = [precision_score(y_te, proba >= t, zero_division=0) for t in ts]
rec = [recall_score(y_te, proba >= t, zero_division=0) for t in ts]
f1 = [f1_score(y_te, proba >= t, zero_division=0) for t in ts]

best_t = ts[int(np.argmax(f1))]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(ts, prec, label='precision', lw=2)
ax.plot(ts, rec, label='recall', lw=2)
ax.plot(ts, f1, label='F1', lw=2, ls='--', color='#c0392b')
ax.axvline(0.5, color='grey', ls=':', label='default 0.5')
ax.axvline(best_t, color='green', ls=':', label=f'best F1 @ {best_t:.2f}')
ax.set_xlabel('decision threshold'); ax.set_ylabel('score')
ax.set_title('Threshold slider - precision/recall is a trade you choose')
ax.legend(); ax.grid(alpha=.3)
fig.savefig(OUT / 'a1_threshold_sweep.png', dpi=130, bbox_inches='tight')
plt.show()

for t in (0.5, best_t):
    tn, fp, fn, tp = confusion_matrix(y_te, proba >= t).ravel()
    print(f'threshold {t:.2f} -> TP={tp:3d} FP={fp:3d} FN={fn:3d} TN={tn:3d} | '
          f'precision={precision_score(y_te, proba>=t, zero_division=0):.3f} '
          f'recall={recall_score(y_te, proba>=t, zero_division=0):.3f}')

threshold 0.50 -> TP= 35 FP=  5 FN= 19 TN=341 | precision=0.875 recall=0.648
threshold 0.45 -> TP= 37 FP=  7 FN= 17 TN=339 | precision=0.841 recall=0.685


/var/folders/c4/t_n58l316yl308hxjf5jlwsh0000gn/T/ipykernel_18217/1166567588.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Discussion (the brief's questions)

**"Can the tool minimise the issues from running algorithms that affect humans?"**
Partly. It makes failure *visible* - the slice table (Probe 4) is how you catch a model that is
95% accurate overall and 60% accurate on one group. But visibility is not a fix: WIT shows you the
disparity, it does not tell you what is acceptable. That is the gap the **Algorithm Charter** and
the **Data Ethics Advisory Group** (Sowden, Res. 5) exist to fill.

**"Do you need special expertise to read the outputs?"**
To *read* them, no - the counterfactual (Probe 2) is plain English: *this wine, with more alcohol,
flips to good*. To *not be misled* by them, yes. A PDP assumes features move independently; when
they are correlated it draws a curve over wines that could never exist. That is Aha's warning
(Res. 4) in practice: **a confident wrong explanation is worse than no explanation.**

**Carry into Assessment 2:** Probe 0 (model comparison + disagreement count), Probe 4 (slice table)
and Probe 5 (threshold choice) are all directly reusable in the A2 evaluation section.